In [1]:
import pandas as pd
import numpy as np
import commodity_common_functions as CCF
import seaborn as sns
import dataframe_image as dfi
from pathlib import Path
import json
import datetime as dt
from datetime import datetime

# from WindPy import w
# w.start()

retMap = {}


def findRetDistribution(product_id, t):
    if product_id in retMap:
        info = retMap[product_id]
    else:
        SQL_cmd = f"""SELECT TRADE_DATE, PRODUCT_ID, dailyRet_main
            FROM RY_FULL
            WHERE PRODUCT_ID = '{product_id}'
            """
        info = (
            pd.read_sql(sql=SQL_cmd, con=CCF.research)
            .drop_duplicates(subset="TRADE_DATE", keep="first")
            .sort_values("TRADE_DATE")
            .dropna()
        )
        info["cumulative_prod"] = (info["dailyRet_main"] + 1).cumprod()
        info.index = info["TRADE_DATE"]
        retMap[product_id] = info
    rolling_product = info["cumulative_prod"] / info["cumulative_prod"].shift(t).fillna(
        1
    )
    return rolling_product


#### 全量信息，输入为日期，输出为全量信息
def findOptionPrice(date):
    SQL_cmd = f"""SELECT instrument_id, trading_date, close, volume, open_interest,underlying_instr_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    """
    df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    df["underlying_instr_id"] = df["underlying_instr_id"].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )

    datebar = date[:4] + "-" + date[4:6] + "-" + date[6:]
    SQL_cmd = f"""SELECT instrument_id, trading_date, close
    FROM daily_data
    WHERE trading_date = '{datebar}'
    """
    unddf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    unddf["trading_date"] = unddf["trading_date"].apply(
        lambda x: x[:4] + x[5:7] + x[8:10]
    )

    df = pd.merge(
        df,
        unddf,
        left_on=["underlying_instr_id", "trading_date"],
        right_on=["instrument_id", "trading_date"],
        how="left",
    )
    df = df.rename(columns={"close_y": "underlying_close", "close_x": "option_close"})
    df = df.drop(columns=["instrument_id_y"])
    df = df.rename(columns={"instrument_id_x": "instrument_id"})

    info_dict = findUnderlyingOptionInfo(date)

    invalid_rows = []
    error_set = []
    # 遍历每一行
    for index, row in df.iterrows():
        try:
            prod, underlying_id, type_, k = parse_option_contract(row["instrument_id"])
            if (type_ == "c" and row["underlying_close"] > k) or (
                type_ == "p" and row["underlying_close"] < k
            ):
                invalid_rows.append(index)
                continue
            expiredate = info_dict[underlying_id]["expiredate"]
            exchange = info_dict[underlying_id]["exchange_id"]
            commission = info_dict[underlying_id]["open_money_by_vol"]
            margin_ratio = info_dict[underlying_id]["margin_ratio"]
            multi = info_dict[underlying_id]["volume_multiple"]
            tick = info_dict[underlying_id]["price_tick"]
            sector = info_dict[underlying_id]["sector"]
            updownlimit = info_dict[underlying_id]["updownlimit"]

            ttm = len(
                CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
            )
            df.at[index, "product"] = prod
            df.at[index, "sector"] = sector
            df.at[index, "opt_typ"] = type_
            df.at[index, "strike"] = k
            df.at[index, "tdtm"] = ttm
            df.at[index, "cdtm"] = (
                datetime.strptime(expiredate, "%Y%m%d")
                - datetime.strptime(date, "%Y%m%d")
            ).days
            df.at[index, "commission"] = commission
            df.at[index, "multiplier"] = multi
            df.at[index, "tick"] = tick
            df.at[index, "expiredate"] = expiredate
            df.at[index, "updownlimit"] = updownlimit
            df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
            iv = CCF.IV(
                row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
            )
            df.at[index, "iv"] = iv
            df.at[index, "delta"] = CCF.BSMgreeks.delta(
                type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
            )
            otmvalue = (
                row["underlying_close"] * multi - k * multi
                if type_ == "p"
                else k * multi - row["underlying_close"] * multi
            )
            otmvalue = max(otmvalue, 0)
            orimargin = row["underlying_close"] * multi * margin_ratio
            df.at[index, "margin"] = row["option_close"] * multi + max(
                orimargin - 0.5 * otmvalue, 0.5 * orimargin
            )
        except:
            # 若解析失败，记录行索引
            #             print(row['instrument_id'], ValueError)
            invalid_rows.append(index)
            error_set.append((index, row["instrument_id"], ValueError))
    df = df.drop(invalid_rows).dropna().reset_index(drop=True)
    df[["volume", "open_interest", "tdtm", "cdtm", "multiplier"]] = df[
        ["volume", "open_interest", "tdtm", "cdtm", "multiplier"]
    ].astype(int)

    return df


def parse_option_contract(contract):
    # 查找第一个数字的位置
    first_digit_index = next(
        (i for i, char in enumerate(contract) if char.isdigit()), None
    )
    if first_digit_index is None:
        raise ValueError(f"无法解析合约: {contract}，未找到数字。")
    product = contract[:first_digit_index]

    if "-" in contract:
        # 处理形如 a2505-C-3850 的合约
        parts = contract.split("-")
        underlying_id = parts[0]
        type_ = parts[1]
        k = float(parts[2])
    else:
        # 处理形如 zn2505P26000 的合约
        type_index = None
        for i in range(1, len(contract) - 1):
            if (
                contract[i].isalpha()
                and contract[i - 1].isdigit()
                and contract[i + 1].isdigit()
            ):
                type_index = i
                break

        if type_index is None:
            raise ValueError(f"无法解析合约: {contract}，请检查合约格式。")

        underlying_id = contract[:type_index]
        type_ = contract[type_index]
        k = float(contract[type_index + 1 :])

    # 修改 underlying_id
    if underlying_id.startswith("HO"):
        underlying_id = "IH" + underlying_id[2:]
        product = "IH"
    elif underlying_id.startswith("IO"):
        underlying_id = "IF" + underlying_id[2:]
        product = "IF"
    elif underlying_id.startswith("MO"):
        underlying_id = "IM" + underlying_id[2:]
        product = "IM"

    return product, underlying_id, type_.lower(), k


def findUnderlyingOptionInfo(date):
    SQL_cmd = f"""
    SELECT underlying_instr_id,instrument_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    ORDER BY open_interest DESC
    """
    max_open_interest = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    max_open_interest = (
        max_open_interest.groupby("underlying_instr_id").first().reset_index()
    )
    max_open_interest["underlying_instr_id"] = max_open_interest[
        "underlying_instr_id"
    ].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    SQL_cmd = f"""
    SELECT         
        i.underlying_instr_id,
        i.exchange_id,
        i.expiredate,
        i.volume_multiple,
        i.price_tick,
        icr.open_money_by_vol
    FROM instrument i
    LEFT JOIN commission_info icr ON i.instrument_id = icr.instrument_id
    WHERE i.instrument_id IN {tuple(max_open_interest['instrument_id'])}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info["underlying_instr_id"] = instrument_info[
        "underlying_instr_id"
    ].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    instrument_info["open_money_by_vol"] = np.where(
        instrument_info["underlying_instr_id"].str.startswith(("IH", "IF", "IM")),
        15,
        instrument_info["open_money_by_vol"],
    )
    SQL_cmd = f"""
    SELECT instrument_id as underlying_instr_id, short_margin_ratio as margin_ratio, product_id as product
    FROM instrument
    WHERE instrument_id IN {tuple(max_open_interest['underlying_instr_id'])}
    """
    margin_ratio = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    SQL_cmd = f"""
    SELECT PRODUCT_ID as product, WIND_INDUSTRYNAME1 as sector
    FROM WindIndustry
    WHERE PRODUCT_ID IN {tuple(margin_ratio['product'])}
    """
    secinfo = pd.read_sql(sql=SQL_cmd, con=CCF.research)
    secinfo.loc[len(secinfo)] = ["IH", "金融"]
    secinfo.loc[len(secinfo)] = ["IF", "金融"]
    secinfo.loc[len(secinfo)] = ["IC", "金融"]
    secinfo.loc[len(secinfo)] = ["IM", "金融"]
    SQL_cmd = f"""
    SELECT         
    instrument_id,
    prev_settlement,
    limit_up
    FROM daily_data
    WHERE trading_date = '{date[:4]}-{date[4:6]}-{date[6:]}'
    """
    limitupdf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    limitupdf["updownlimit"] = (
        limitupdf["limit_up"] / limitupdf["prev_settlement"] - 1
    ).round(2)
    limitupdf = limitupdf[["instrument_id", "updownlimit"]]
    limitupdf = limitupdf.rename(columns={"instrument_id": "underlying_instr_id"})

    merged_df = pd.merge(
        max_open_interest, instrument_info, on="underlying_instr_id", how="left"
    )
    merged_df = pd.merge(merged_df, margin_ratio, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, limitupdf, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, secinfo, on="product", how="left")
    merged_df = merged_df.drop(columns=["instrument_id"]).dropna()
    merged_dict = merged_df.set_index("underlying_instr_id").T.to_dict("dict")
    for key in merged_dict:
        if merged_dict[key]["exchange_id"] == "CFFEX":
            merged_dict[key]["exchange_id"] = "CFE"
        elif merged_dict[key]["exchange_id"] == "SHFE":
            merged_dict[key]["exchange_id"] = "SHF"
        elif merged_dict[key]["exchange_id"] == "CZCE":
            merged_dict[key]["exchange_id"] = "CZC"
        elif merged_dict[key]["exchange_id"] == "GFEX":
            merged_dict[key]["exchange_id"] = "GFE"
    return merged_dict


def getPayoffSeries(product, dtm, underlying_close, strike, opt_typ):
    pricedis = findRetDistribution(product, dtm) * underlying_close
    if opt_typ == "c":
        payoffser = pricedis - strike
    else:
        payoffser = strike - pricedis
    payoffser[payoffser < 0] = 0
    return payoffser

In [20]:
SQL_cmd = f"""SELECT instrument_id, trading_date, close, volume, open_interest,underlying_instr_id
FROM daily_data_option
WHERE trading_date = '{date}'
"""
df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
df["underlying_instr_id"] = df["underlying_instr_id"].apply(
    lambda x: (
        "IH" + x[2:]
        if x.startswith("HO")
        else (
            "IF" + x[2:]
            if x.startswith("IO")
            else ("IM" + x[2:] if x.startswith("MO") else x)
        )
    )
)

datebar = date[:4] + "-" + date[4:6] + "-" + date[6:]
SQL_cmd = f"""SELECT instrument_id, trading_date, close
FROM daily_data
WHERE trading_date = '{datebar}'
"""
unddf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
unddf["trading_date"] = unddf["trading_date"].apply(
    lambda x: x[:4] + x[5:7] + x[8:10]
)

df = pd.merge(
    df,
    unddf,
    left_on=["underlying_instr_id", "trading_date"],
    right_on=["instrument_id", "trading_date"],
    how="left",
)
df = df.rename(columns={"close_y": "underlying_close", "close_x": "option_close"})
df = df.drop(columns=["instrument_id_y"])
df = df.rename(columns={"instrument_id_x": "instrument_id"})

info_dict = findUnderlyingOptionInfo(date)

In [18]:
invalid_rows = []
error_set = []
# 遍历每一行
for index, row in df.iterrows():
    # try:
        prod, underlying_id, type_, k = parse_option_contract(row["instrument_id"])
        if (type_ == "c" and row["underlying_close"] > k) or (
            type_ == "p" and row["underlying_close"] < k
        ):
            invalid_rows.append(index)
            continue
        expiredate = info_dict[underlying_id]["expiredate"]
        exchange = info_dict[underlying_id]["exchange_id"]
        commission = info_dict[underlying_id]["open_money_by_vol"]
        margin_ratio = info_dict[underlying_id]["margin_ratio"]
        multi = info_dict[underlying_id]["volume_multiple"]
        tick = info_dict[underlying_id]["price_tick"]
        sector = info_dict[underlying_id]["sector"]
        updownlimit = info_dict[underlying_id]["updownlimit"]

        ttm = len(
            CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
        )
        df.at[index, "product"] = prod
        df.at[index, "sector"] = sector
        df.at[index, "opt_typ"] = type_
        df.at[index, "strike"] = k
        df.at[index, "tdtm"] = ttm
        df.at[index, "cdtm"] = (
            datetime.strptime(expiredate, "%Y%m%d")
            - datetime.strptime(date, "%Y%m%d")
        ).days
        df.at[index, "commission"] = commission
        df.at[index, "multiplier"] = multi
        df.at[index, "tick"] = tick
        df.at[index, "expiredate"] = expiredate
        df.at[index, "updownlimit"] = updownlimit
        df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
        iv = CCF.IV(
            row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
        )
        print( prod, underlying_id, type_, k, iv)
        print(row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_)
        print()
        df.at[index, "iv"] = iv
        df.at[index, "delta"] = CCF.BSMgreeks.delta(
            type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
        )
        otmvalue = (
            row["underlying_close"] * multi - k * multi
            if type_ == "p"
            else k * multi - row["underlying_close"] * multi
        )
        otmvalue = max(otmvalue, 0)
        orimargin = row["underlying_close"] * multi * margin_ratio
        df.at[index, "margin"] = row["option_close"] * multi + max(
            orimargin - 0.5 * otmvalue, 0.5 * orimargin
        )
    # except:
    #     # 若解析失败，记录行索引
    #     #             print(row['instrument_id'], ValueError)
    #     invalid_rows.append(index)
    #     error_set.append((index, row["instrument_id"], ValueError))
# df = df.drop(invalid_rows).dropna().reset_index(drop=True)

a a2507 c 4150.0 0.13317285703271964
72.0 4116.0 4150.0 0.16049382716049382 0 c

a a2507 c 4200.0 0.13356644955584057
53.0 4116.0 4200.0 0.16049382716049382 0 c

a a2507 c 4250.0 0.13567339238798823
39.0 4116.0 4250.0 0.16049382716049382 0 c

a a2507 c 4300.0 0.14423050739717347
31.5 4116.0 4300.0 0.16049382716049382 0 c

a a2507 c 4350.0 0.14518712870604505
22.5 4116.0 4350.0 0.16049382716049382 0 c

a a2507 c 4400.0 0.14948794436452878
17.0 4116.0 4400.0 0.16049382716049382 0 c

a a2507 c 4450.0 0.1525266929179148
12.5 4116.0 4450.0 0.16049382716049382 0 c

a a2507 c 4500.0 0.1626329204474596
11.0 4116.0 4500.0 0.16049382716049382 0 c

a a2507 c 4550.0 0.16468232903473082
8.0 4116.0 4550.0 0.16049382716049382 0 c

a a2507 c 4600.0 0.17067464306738175
6.5 4116.0 4600.0 0.16049382716049382 0 c

a a2507 c 4650.0 0.1776509116317883
5.5 4116.0 4650.0 0.16049382716049382 0 c

a a2507 c 4750.0 0.1866091232585175
3.5 4116.0 4750.0 0.16049382716049382 0 c

a a2507 c 4800.0 0.1977599827974291


ValueError: cannot convert float NaN to integer

In [6]:
date = (dt.date.today()).strftime(format="%Y%m%d")
# date = "20250414"
# 读取期权价格数据
optdf = findOptionPrice(date)
# 设置期权价格数据的索引
optdf.index = optdf["instrument_id"]
# 调整保证金
optdf["margin"] = optdf["margin"] * 1.1
optdf["margin"] = optdf["margin"].round(0)
# 计算期权的收益
payoffmap = {}

def calculate_payoff(row):
    ins_id = row["instrument_id"]
    product = row["product"]
    dtm = row["tdtm"]
    underlying_close = row["underlying_close"]
    strike = row["strike"]
    opt_typ = row["opt_typ"]
    payoff = (
        getPayoffSeries(product, dtm, underlying_close, strike, opt_typ)
        * row["multiplier"]
    )
    payoffmap[ins_id] = payoff
    return pd.Series(
        [payoff.mean(), payoff.quantile(0.95), payoff.max(), len(payoff)]
    )

optdf["NumLimit"] = (
    np.abs(np.log(optdf["strike"] / optdf["underlying_close"]))
    / optdf["updownlimit"]
).round(2)
optdf["TradingValue"] = (
    optdf["option_close"] * optdf["multiplier"] * optdf["volume"]
)
optdf[["ExpectedPayoff", "Q95Payoff", "MaxPayoff", "DaysInSample"]] = optdf.apply(
    calculate_payoff, axis=1
)
# 计算期权的信用收取
optdf["CreditCollected"] = optdf["option_close"] * optdf["multiplier"]
# 计算期权的预期承保费
optdf["ExpectedLotPremium"] = optdf["CreditCollected"] - optdf["ExpectedPayoff"]
# 计算期权的理想收益
optdf["IdealRet(C)"] = (
    (optdf["CreditCollected"] - optdf["commission"])
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的预期边际收益
optdf["EMarginRet(C)"] = (
    (optdf["ExpectedLotPremium"] - optdf["commission"] * 2)
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的95%边际收益
optdf["95MarginRet(C)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["Q95Payoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的最差边际收益
optdf["WorstMarginRet(C)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["MaxPayoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)

optdf["IdealRet(T)"] = (
    (optdf["CreditCollected"] - optdf["commission"])
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的预期边际收益
optdf["EMarginRet(T)"] = (
    (optdf["ExpectedLotPremium"] - optdf["commission"] * 2)
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的95%边际收益
optdf["95MarginRet(T)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["Q95Payoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的最差边际收益
optdf["WorstMarginRet(T)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["MaxPayoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)

# 删除空值
# optdf = optdf.dropna()

In [8]:
optdf['product'].unique()

array(['a', 'ag', 'al', 'ao', 'AP', 'au', 'b', 'br', 'c', 'CF', 'CJ',
       'cs', 'cu', 'eb', 'eg', 'FG', 'i', 'IF', 'jd', 'l', 'lc', 'lh',
       'm', 'MA', 'ni', 'OI', 'p', 'pb', 'PF', 'pg', 'PK', 'pp', 'PX',
       'rb', 'RM', 'ru', 'SA', 'sc', 'SF', 'SH', 'si', 'SM', 'sn', 'SR',
       'TA', 'UR', 'v', 'y', 'zn'], dtype=object)

In [15]:

retMap = {}


def findRetDistribution(product_id, t):
    if product_id in retMap:
        info = retMap[product_id]
    else:
        info = ak.fund_etf_hist_em(symbol=product_id, period="daily", start_date="20000101", end_date="20500101", adjust="hfq")[['日期','收盘']]
        info.columns = ['date','close']
        info['dailyRet_main'] = info['close'].pct_change()
        info["cumulative_prod"] = (info["dailyRet_main"] + 1).cumprod()
        info["date"] = info["date"].apply(lambda x: x[:4] + x[5:7] + x[8:10])
        info.index = info["date"]
        retMap[product_id] = info
    rolling_product = info["cumulative_prod"] / info["cumulative_prod"].shift(t).fillna(
        1
    )
    return rolling_product

close_dict = {}
def findETFclose(date, code):
    if(code in close_dict):
        temp = close_dict[code]
    else:
        temp = ak.fund_etf_hist_em(symbol=code, period="daily")
        temp['日期'] = temp['日期'].apply(lambda x: x[:4] + x[5:7] + x[8:])
        close_dict[code] = temp
    # print(ak.fund_etf_hi'st_em(symbol=code, period="daily", end_date=date))
    return float(temp[temp['日期']<=date].iloc[-1]['收盘'])

def findOptionPrice(date):
    SQL_cmd = f"""SELECT instrument_id, trading_date, close, volume, underlying_instr_id
    FROM daily_data_ut
    WHERE trading_date = '{date}'
    """
    df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    df_merged = df[df["instrument_id"].str.len() > 6]
    df_merged = df_merged.rename(columns={"close": "option_close"})
    SQL_cmd = f"""
    SELECT instrument_id, strike_price as strike, options_type as opt_typ, exchange_id, expiredate, volume_multiple as multiplier, price_tick
    FROM instrument
    WHERE instrument_id IN {tuple(df_merged['instrument_id'])}
    """
    info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    info["sector"] = "ETF"
    info["commission"] = 1.7 / 2
    info["updownlimit"] = 0.1
    df = pd.merge(df_merged, info, on="instrument_id", how="left")
    invalid_rows = []
    error_set = []
    for index, row in df.iterrows():
        try:
            type_ = "c" if row["opt_typ"] == '1' else "p"
            k = row["strike"]
            underlying_id = row["underlying_instr_id"]
            und_close = findETFclose(date, underlying_id)
            df.at[index, "underlying_close"] = und_close
            df.at[index, "opt_typ"] = type_
            if (type_ == "c" and und_close > k) or (type_ == "p" and und_close < k):
                invalid_rows.append(index)
                continue
            expiredate = row["expiredate"]
            ttm = len(
                CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
            )
            df.at[index, "tdtm"] = ttm
            df.at[index, "cdtm"] = (
                datetime.strptime(expiredate, "%Y%m%d") - datetime.strptime(date, "%Y%m%d")
            ).days
            exchange = "SZ" if row["exchange_id"] == "SZSE" else "SH"
            df.at[index, "windcode"] = row["instrument_id"] + "." + exchange
            if underlying_id == "510300":
                con_undcode = "沪深300"
            elif underlying_id == "510500":
                con_undcode = "中证500"
            elif underlying_id == "510050":
                con_undcode = "上证50"
            elif underlying_id == "159915":
                con_undcode = "创业板"
            elif underlying_id == "588000":
                con_undcode = "科创50"
            elif underlying_id == "159901":
                con_undcode = "深证100"
            conepire = expiredate[2:6]
            df.at[index, "ezCode"] = f"{con_undcode}_{conepire}_{k}{type_}"
            iv = CCF.IV(row["option_close"], und_close, k, ttm / 243, 0, type_)
            df.at[index, "iv"] = iv
            df.at[index, "delta"] = CCF.BSMgreeks.delta(
                type_, und_close, k, ttm / 243, 0, iv, 0
            )
            otmvalue = (
                und_close - k if type_ == "p" else k - und_close
            )
            multi = row["multiplier"]
            otmvalue = max(otmvalue, 0)
            # 计算保证金
            if type_ == "c":  # 看涨期权
                margin_rate_1 = 0.12 * und_close - otmvalue
                margin_rate_2 = 0.07 * und_close
                margin = (row["option_close"] + max(margin_rate_1, margin_rate_2)) * multi
            else:  # 看跌期权
                margin_rate_1 = 0.12 * und_close - otmvalue
                margin_rate_2 = 0.07 * k
                margin = min(row["option_close"] + max(margin_rate_1, margin_rate_2), k) * multi
            # orimargin = row["underlying_close"] * multi * 0.12
            df.at[index, "margin"] = margin
        except:
            # 若解析失败，记录行索引
            #             print(row['instrument_id'], ValueError)
            invalid_rows.append(index)
            error_set.append((index, row["instrument_id"], ValueError))
    df = df.drop(invalid_rows).dropna().reset_index(drop=True)
    df[["volume",  "tdtm", "cdtm", "multiplier"]] = df[
        ["volume",  "tdtm", "cdtm", "multiplier"]
    ].astype(int)
    return df


def getPayoffSeries(product, dtm, underlying_close, strike, opt_typ):
    pricedis = findRetDistribution(product, dtm) * underlying_close
    if opt_typ == "c":
        payoffser = pricedis - strike
    else:
        payoffser = strike - pricedis
    payoffser[payoffser < 0] = 0
    return payoffser

def ETF_Option_Raw_Data(wind_code: str, tradingdate: str):
    """

    :param wind_code:
    :param tradingdate:
    :return: dataframe 包含字段 id, seq_no, Bid1, Ask1, MidPrice
    """
    future_path = CCF.Ftp_Path(tradingdate)
    option_data = f'{future_path}ut_std_data/{tradingdate}/{wind_code}.csv'
    # print(option_data)
    OptionData = pd.read_csv(option_data, usecols=['id', 'seq_no', 'bid_price1', 'ask_price1'],
                             dtype={'id': object, 'seq_no': np.int64, 'bid_price1': np.float64,
                                    'ask_price1': np.float64},
                             escapechar='/', na_values=r"\N")

    OptionData.rename(columns={'bid_price1': 'Bid1', 'ask_price1': 'Ask1'}, inplace=True)
    OptionData.loc[:, 'MidPrice'] = 0.5 * (OptionData['Bid1'] + OptionData['Ask1'])

    return OptionData

In [4]:
date = (dt.date.today()).strftime(format="%Y%m%d")
# date = "20250414"
# 读取期权价格数据
optdf = findOptionPrice(date)
# 设置期权价格数据的索引
optdf.index = optdf["ezCode"]
# 调整保证金
optdf["margin"] = optdf["margin"] * 1.1
optdf["margin"] = optdf["margin"].round(0)
# 计算期权的收益
payoffmap = {}

def calculate_payoff(row):
    ins_id = row["instrument_id"]
    product = row["underlying_instr_id"]
    dtm = row["tdtm"]
    underlying_close = row["underlying_close"]
    strike = row["strike"]
    opt_typ = row["opt_typ"]
    payoff = (
        getPayoffSeries(product, dtm, underlying_close, strike, opt_typ)
        * row["multiplier"]
    )
    payoffmap[ins_id] = payoff
    return pd.Series(
        [payoff.mean(), payoff.quantile(0.95), payoff.max(), len(payoff)]
    )

optdf["NumLimit"] = (
    np.abs(np.log(optdf["strike"] / optdf["underlying_close"]))
    / optdf["updownlimit"]
).round(2)
optdf["TradingValue"] = (
    optdf["option_close"] * optdf["multiplier"] * optdf["volume"]
)
optdf[["ExpectedPayoff", "Q95Payoff", "MaxPayoff", "DaysInSample"]] = optdf.apply(
    calculate_payoff, axis=1
)
# 计算期权的信用收取
optdf["CreditCollected"] = optdf["option_close"] * optdf["multiplier"]
# 计算期权的预期承保费
optdf["ExpectedLotPremium"] = optdf["CreditCollected"] - optdf["ExpectedPayoff"]
# 计算期权的理想收益
optdf["IdealRet(C)"] = (
    (optdf["CreditCollected"] - optdf["commission"])
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的预期边际收益
optdf["EMarginRet(C)"] = (
    (optdf["ExpectedLotPremium"] - optdf["commission"] * 2)
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的95%边际收益
optdf["95MarginRet(C)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["Q95Payoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的最差边际收益
optdf["WorstMarginRet(C)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["MaxPayoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)

optdf["IdealRet(T)"] = (
    (optdf["CreditCollected"] - optdf["commission"])
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的预期边际收益
optdf["EMarginRet(T)"] = (
    (optdf["ExpectedLotPremium"] - optdf["commission"] * 2)
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的95%边际收益
optdf["95MarginRet(T)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["Q95Payoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的最差边际收益
optdf["WorstMarginRet(T)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["MaxPayoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)

In [18]:
optdf = optdf.dropna()

# 复制一份期权数据，为筛选后结果
std_df = optdf.copy()
# 筛选出预期边际收益大于等于0.05的数据
std_df = std_df[std_df["EMarginRet(C)"] >= 0.05]
# 筛选出delta绝对值小于0.1的数据
std_df = std_df[np.abs(std_df["delta"]) < 0.1]
# 筛选出dtm小于等于20的数据
std_df = std_df[std_df["tdtm"] <= 35]
# 筛选出持仓量大于等于200的数据
std_df = std_df[std_df["volume"] >= 500]
# 筛选出最大收益为0且样本天数大于等于500的数据
std_df = std_df[(std_df["MaxPayoff"] == 0) & (std_df["DaysInSample"] >= 500)]

#  输入为一行optdf，输出该行在该交易日14:55时间点的ask和bid
def findOptionQuote(row):
    date = row["trading_date"]
    ins = row["instrument_id"]
    try:
        row = ETF_Option_Raw_Data(ins, date).iloc[-601]
        return float(row["Ask1"]), float(row["Bid1"])
    except:
        return np.nan, np.nan

std_df["bid"] = std_df.apply(lambda x: findOptionQuote(x)[1], axis=1)
# 生成excel数据
exceldf = (
    std_df[
        [
            "underlying_instr_id",
            "option_close",
            "sector",
            "NumLimit",
            "EMarginRet(C)",
            "EMarginRet(T)",
            "margin",
            "tdtm",
            "cdtm",
            "bid"
        ]
    ]
    .dropna()
    .reset_index()
)
exceldf["EMarginRet(C)"] = (exceldf["EMarginRet(C)"] * 100).round(2)
exceldf["EMarginRet(T)"] = (exceldf["EMarginRet(T)"] * 100).round(2)
undrownum = exceldf.groupby("underlying_instr_id").size()
undmeanret = (
    exceldf.groupby("underlying_instr_id")["EMarginRet(T)"]
    .mean()
    .sort_values(ascending=False)
)
exceldf = exceldf.groupby(["underlying_instr_id", "instrument_id"]).first()
result = []
cnt = 0
sett = []
for und in undmeanret.index:
    thisnum = undrownum[und]
    if thisnum + cnt > 120:
        result.append(exceldf.loc[(sett, slice(None)), :])
        sett = [und]
        cnt = thisnum
    else:
        sett.append(und)
        cnt += thisnum
result.append(exceldf.loc[(sett, slice(None)), :])

# 生成json数据
jsondf = std_df.reset_index(drop=True)
jsonoutput = []
for ind, row in jsondf.iterrows():
    rowjs = {}
    idd = f"{date[2:]}#{ind + 1}"
    rowjs["_id"] = idd
    rowjs["idnum"] = ind + 1
    rowjs["create_date"] = date
    rowjs["underlying_id"] = row["underlying_instr_id"]
    rowjs["sector"] = row["sector"]
    rowjs["margin_req"] = row["margin"]
    rowjs["expire_date"] = row["expiredate"]
    rowjs["STG_TYP"] = ["NK_SELL"]
    rowjs["HEDG_MTD"] = ["StopLoss"]
    rowjs["option_structure"] = {row["instrument_id"]: -1}
    rowjs["risk_params"] = {
        "limit_price": row["option_close"] * -1,
        "ExpectedPayoff": row["ExpectedPayoff"],
        "EMarginRet": row["EMarginRet(T)"],
    }
    jsonoutput.append(rowjs)

,instrument_id,trading_date,option_close,volume,underlying_instr_id,strike,opt_typ,exchange_id,expiredate,multiplier,...,ExpectedLotPremium,IdealRet(C),EMarginRet(C),95MarginRet(C),WorstMarginRet(C),IdealRet(T),EMarginRet(T),95MarginRet(T),WorstMarginRet(T),bid
instrument_id,,,,,,,,,,,,,,,,,,,,,
10009032,10009032,20250417,0.0003,4215,588000,0.90,p,SSE,20250423,10000,...,3.0,0.187919,0.113625,0.113625,0.113625,0.187662,0.113470,0.113470,0.113470,0.0003
10009144,10009144,20250417,0.0003,6135,588000,0.85,p,SSE,20250423,10000,...,3.0,0.198772,0.120187,0.120187,0.120187,0.198499,0.120023,0.120023,0.120023,0.0002
10009201,10009201,20250417,0.0010,8745,588000,0.75,p,SSE,20250528,10000,...,10.0,0.138298,0.125450,0.125450,0.125450,0.145191,0.131703,0.131703,0.131703,0.0010
10009202,10009202,20250417,0.0019,651,588000,0.80,p,SSE,20250528,10000,...,19.0,0.253657,0.241777,0.241777,0.241777,0.266299,0.253828,0.253828,0.253828,0.0018


In [5]:

date = "20250411"
SQL_cmd = f"""SELECT instrument_id, trading_date, close, volume, underlying_instr_id
FROM daily_data_ut
WHERE trading_date = '{date}'
"""
df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
df_merged = df[df["instrument_id"].str.len() > 6]
# 修改列名 close 为 option_close
df_merged = df_merged.rename(columns={"close": "option_close"})
# df_und = df[df['instrument_id'].str.len() <= 6]
# df_merged = pd.merge(df_option, df_und[['instrument_id','close']], left_on='underlying_instr_id', right_on='instrument_id', how='left')
# df_merged = df_merged[['instrument_id_x','trading_date','close_x','volume','underlying_instr_id','close_y']]
# df_merged.columns = ['instrument_id','trading_date','option_close','volume','underlying_instr_id','underlying_close']
SQL_cmd = f"""
SELECT instrument_id, strike_price as strike, options_type as opt_typ, exchange_id, expiredate, volume_multiple as multiplier, price_tick
FROM instrument
WHERE instrument_id IN {tuple(df_merged['instrument_id'])}
"""
info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
# info['opt_typ'] = info['opt_typ'].apply(lambda x: 'c' if x == '1' else 'p')
# info['exchange_id'] = info['exchange_id'].apply(lambda x: 'SZ' if x == 'SSE' else 'SH')
info["sector"] = "ETF"
info["commission"] = 1.7 / 2
info["updownlimit"] = 0.1
df = pd.merge(df_merged, info, on="instrument_id", how="left")
invalid_rows = []
error_set = []
# 遍历每一行
for index, row in df.iterrows():
    try:
        type_ = "c" if row["opt_typ"] == '1' else "p"
        k = row["strike"]
        underlying_id = row["underlying_instr_id"]
        und_close = findETFclose(date, underlying_id)
        df.at[index, "underlying_close"] = und_close
        df.at[index, "opt_typ"] = type_
        if (type_ == "c" and und_close > k) or (type_ == "p" and und_close < k):
            invalid_rows.append(index)
            continue
        expiredate = row["expiredate"]
        ttm = len(
            CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
        )
        df.at[index, "tdtm"] = ttm
        df.at[index, "cdtm"] = (
            datetime.strptime(expiredate, "%Y%m%d") - datetime.strptime(date, "%Y%m%d")
        ).days
        exchange = "SZ" if row["exchange_id"] == "SZSE" else "SH"
        df.at[index, "windcode"] = row["instrument_id"] + "." + exchange
        if underlying_id == "510300":
            con_undcode = "沪深300"
        elif underlying_id == "510500":
            con_undcode = "中证500"
        elif underlying_id == "510050":
            con_undcode = "上证50"
        elif underlying_id == "159915":
            con_undcode = "创业板"
        elif underlying_id == "588000":
            con_undcode = "科创50"
        elif underlying_id == "159901":
            con_undcode = "深证100"
        conepire = expiredate[2:6]
        df.at[index, "ezCode"] = f"{con_undcode}_{conepire}_{k}{type_}"
        iv = CCF.IV(row["option_close"], und_close, k, ttm / 243, 0, type_)
        df.at[index, "iv"] = iv
        df.at[index, "delta"] = CCF.BSMgreeks.delta(
            type_, und_close, k, ttm / 243, 0, iv, 0
        )
        otmvalue = (
            und_close - k if type_ == "p" else k - und_close
        )
        multi = row["multiplier"]
        otmvalue = max(otmvalue, 0)
        # 计算保证金
        if type_ == "c":  # 看涨期权
            margin_rate_1 = 0.12 * und_close - otmvalue
            margin_rate_2 = 0.07 * und_close
            margin = (row["option_close"] + max(margin_rate_1, margin_rate_2)) * multi
        else:  # 看跌期权
            margin_rate_1 = 0.12 * und_close - otmvalue
            margin_rate_2 = 0.07 * k
            margin = min(row["option_close"] + max(margin_rate_1, margin_rate_2), k) * multi
        # orimargin = row["underlying_close"] * multi * 0.12
        df.at[index, "margin"] = margin
    except:
        # 若解析失败，记录行索引
        #             print(row['instrument_id'], ValueError)
        invalid_rows.append(index)
        error_set.append((index, row["instrument_id"], ValueError))
df = df.drop(invalid_rows).dropna().reset_index(drop=True)
df[["volume",  "tdtm", "cdtm", "multiplier"]] = df[
    ["volume",  "tdtm", "cdtm", "multiplier"]
].astype(int)


#         ttm = len(
#             CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
#         )
#         df.at[index, "product"] = prod
#         df.at[index, "sector"] = sector
#         df.at[index, "opt_typ"] = type_
#         df.at[index, "strike"] = k
#         df.at[index, "tdtm"] = ttm
#         df.at[index, "cdtm"] = (
#             datetime.strptime(expiredate, "%Y%m%d")
#             - datetime.strptime(date, "%Y%m%d")
#         ).days
#         df.at[index, "commission"] = commission
#         df.at[index, "multiplier"] = multi
#         df.at[index, "tick"] = tick
#         df.at[index, "expiredate"] = expiredate
#         df.at[index, "updownlimit"] = updownlimit
#         df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
#         iv = CCF.IV(
#             row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
#         )
#         df.at[index, "iv"] = iv
#         df.at[index, "delta"] = CCF.BSMgreeks.delta(
#             type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
#         )
#         otmvalue = (
#             row["underlying_close"] * multi - k * multi
#             if type_ == "p"
#             else k * multi - row["underlying_close"] * multi
#         )
#         otmvalue = max(otmvalue, 0)
#         orimargin = row["underlying_close"] * multi * margin_ratio
#         df.at[index, "margin"] = row["option_close"] * multi + max(
#             orimargin - 0.5 * otmvalue, 0.5 * orimargin
#         )

In [ ]:
df

In [19]:
CCF.IV(0.2533, 1.061, 0.8, 0.20164609053497942, 0, 'p')

nan

In [16]:
1.061-0.8

0.2609999999999999

In [5]:
invalid_rows

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [7]:
date = '20250411'
SQL_cmd = f"""SELECT instrument_id, trading_date, close, volume, underlying_instr_id
FROM daily_data_ut
WHERE trading_date = '{date}'
"""
df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
# datebar = date[:4] + "-" + date[4:6] + "-" + date[6:]
# SQL_cmd = f"""SELECT instrument_id, trading_date, close
# FROM daily_data
# WHERE trading_date = '{datebar}'
# """
# unddf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
# unddf["trading_date"] = unddf["trading_date"].apply(
#     lambda x: x[:4] + x[5:7] + x[8:10]
# )

# df = pd.merge(
#     df,
#     unddf,
#     left_on=["underlying_instr_id", "trading_date"],
#     right_on=["instrument_id", "trading_date"],
#     how="left",
# )
# df = df.rename(columns={"close_y": "underlying_close", "close_x": "option_close"})
# df = df.drop(columns=["instrument_id_y"])
# df = df.rename(columns={"instrument_id_x": "instrument_id"})

# info_dict = findUnderlyingOptionInfo(date)

# invalid_rows = []
# error_set = []
# # 遍历每一行
# for index, row in df.iterrows():
#     try:
#         prod, underlying_id, type_, k = parse_option_contract(row["instrument_id"])
#         if (type_ == "c" and row["underlying_close"] > k) or (
#             type_ == "p" and row["underlying_close"] < k
#         ):
#             invalid_rows.append(index)
#             continue
#         expiredate = info_dict[underlying_id]["expiredate"]
#         exchange = info_dict[underlying_id]["exchange_id"]
#         commission = info_dict[underlying_id]["open_money_by_vol"]
#         margin_ratio = info_dict[underlying_id]["margin_ratio"]
#         multi = info_dict[underlying_id]["volume_multiple"]
#         tick = info_dict[underlying_id]["price_tick"]
#         sector = info_dict[underlying_id]["sector"]
#         updownlimit = info_dict[underlying_id]["updownlimit"]

#         ttm = len(
#             CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
#         )
#         df.at[index, "product"] = prod
#         df.at[index, "sector"] = sector
#         df.at[index, "opt_typ"] = type_
#         df.at[index, "strike"] = k
#         df.at[index, "tdtm"] = ttm
#         df.at[index, "cdtm"] = (
#             datetime.strptime(expiredate, "%Y%m%d")
#             - datetime.strptime(date, "%Y%m%d")
#         ).days
#         df.at[index, "commission"] = commission
#         df.at[index, "multiplier"] = multi
#         df.at[index, "tick"] = tick
#         df.at[index, "expiredate"] = expiredate
#         df.at[index, "updownlimit"] = updownlimit
#         df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
#         iv = CCF.IV(
#             row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
#         )
#         df.at[index, "iv"] = iv
#         df.at[index, "delta"] = CCF.BSMgreeks.delta(
#             type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
#         )
#         otmvalue = (
#             row["underlying_close"] * multi - k * multi
#             if type_ == "p"
#             else k * multi - row["underlying_close"] * multi
#         )
#         otmvalue = max(otmvalue, 0)
#         orimargin = row["underlying_close"] * multi * margin_ratio
#         df.at[index, "margin"] = row["option_close"] * multi + max(
#             orimargin - 0.5 * otmvalue, 0.5 * orimargin
#         )
#     except:
#         # 若解析失败，记录行索引
#         #             print(row['instrument_id'], ValueError)
#         invalid_rows.append(index)
#         error_set.append((index, row["instrument_id"], ValueError))
# df = df.drop(invalid_rows).dropna().reset_index(drop=True)
# df[["volume", "open_interest", "tdtm", "cdtm", "multiplier"]] = df[
#     ["volume", "open_interest", "tdtm", "cdtm", "multiplier"]
# ].astype(int)

In [4]:
retMap = {}


def findRetDistribution(product_id, t):
    if product_id in retMap:
        info = retMap[product_id]
    else:
        info = ak.fund_etf_hist_em(symbol=product_id, period="daily", start_date="20000101", end_date="20500101", adjust="hfq")[['日期','收盘']]
        info.columns = ['date','close']
        info['dailyRet_main'] = info['close'].pct_change()
        info["cumulative_prod"] = (info["dailyRet_main"] + 1).cumprod()
        info["date"] = info["date"].apply(lambda x: x[:4] + x[5:7] + x[8:10])
        info.index = info["date"]
        retMap[product_id] = info
    rolling_product = info["cumulative_prod"] / info["cumulative_prod"].shift(t).fillna(
        1
    )
    return rolling_product


def findUnderlyingOptionInfo(date):
    SQL_cmd = f"""
    SELECT underlying_instr_id,instrument_id
    FROM daily_data_ut
    WHERE trading_date = '{date}' AND underlying_instr_id != ''
    ORDER BY open_interest DESC
    """
    max_open_interest = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    max_open_interest = (
        max_open_interest.groupby("underlying_instr_id").first().reset_index()
    )

    SQL_cmd = f"""
    SELECT         
        i.underlying_instr_id,
        i.exchange_id,
        i.expiredate,
        i.volume_multiple,
        i.price_tick
    FROM instrument i
    WHERE i.instrument_id IN {tuple(max_open_interest['instrument_id'])}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info['open_money_by_vol'] = 1.7/2
    instrument_info['product'] =  instrument_info['underlying_instr_id']
    instrument_info['margin_ratio'] =  0.12
    instrument_info['sector'] =  'ETF'
    instrument_info['updownlimit'] = 0.1
    
    # SQL_cmd = f"""
    # SELECT instrument_id as underlying_instr_id, short_margin_ratio as margin_ratio, product_id as product
    # FROM instrument
    # WHERE instrument_id IN {tuple(max_open_interest['underlying_instr_id'])}
    # """
    # margin_ratio = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    # SQL_cmd = f"""
    # SELECT PRODUCT_ID as product, WIND_INDUSTRYNAME1 as sector
    # FROM WindIndustry
    # WHERE PRODUCT_ID IN {tuple(margin_ratio['product'])}
    # """
    # secinfo = pd.read_sql(sql=SQL_cmd, con=CCF.research)
    # secinfo.loc[len(secinfo)] = ['IH','金融']
    # secinfo.loc[len(secinfo)] = ['IF','金融']
    # secinfo.loc[len(secinfo)] = ['IC','金融']
    # secinfo.loc[len(secinfo)] = ['IM','金融']
    # SQL_cmd = f"""
    # SELECT         
    # instrument_id,
    # prev_settlement,
    # limit_up
    # FROM daily_data
    # WHERE trading_date = '{date[:4]}-{date[4:6]}-{date[6:]}'
    # """
    # limitupdf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    # limitupdf["updownlimit"] = (
    #     limitupdf["limit_up"] / limitupdf["prev_settlement"] - 1
    # ).round(2)
    # limitupdf = limitupdf[["instrument_id", "updownlimit"]]
    # limitupdf = limitupdf.rename(columns={"instrument_id": "underlying_instr_id"})

    merged_df = pd.merge(
        max_open_interest, instrument_info, on="underlying_instr_id", how="left"
    )
    # merged_df = pd.merge(merged_df, margin_ratio, on="underlying_instr_id", how="left")
    # merged_df = pd.merge(merged_df, limitupdf, on="underlying_instr_id", how="left")
    # merged_df = pd.merge(merged_df, secinfo, on="product", how="left")
    merged_df = merged_df.drop(columns=["instrument_id"]).dropna()
    return merged_df
    merged_dict = merged_df.set_index("underlying_instr_id").T.to_dict("dict")
    for key in merged_dict:
        if merged_dict[key]["exchange_id"] == "SZSE":
            merged_dict[key]["exchange_id"] = "SZ"
        elif merged_dict[key]["exchange_id"] == "SSE":
            merged_dict[key]["exchange_id"] = "SH"
    return merged_dict



In [5]:
findUnderlyingOptionInfo('20250411')

,underlying_instr_id,exchange_id,expiredate,volume_multiple,price_tick,open_money_by_vol,product,margin_ratio,sector,updownlimit
0,159901,SZSE,20250625,10000,0.0001,0.85,159901,0.12,ETF,0.1
1,159915,SZSE,20250625,10000,0.0001,0.85,159915,0.12,ETF,0.1
2,510050,SSE,20250625,10000,0.0001,0.85,510050,0.12,ETF,0.1
3,510300,SSE,20250625,10000,0.0001,0.85,510300,0.12,ETF,0.1
4,510500,SSE,20250625,10000,0.0001,0.85,510500,0.12,ETF,0.1
5,588000,SSE,20250625,10000,0.0001,0.85,588000,0.12,ETF,0.1


In [5]:
findRetDistribution('159915',5)

date
20111209         NaN
20111212    0.991217
20111213    0.963614
20111214    0.953576
20111215    0.943538
              ...   
20250411    0.933399
20250414    1.067606
20250415    1.049335
20250416    1.025782
20250417    1.010735
Name: cumulative_prod, Length: 3241, dtype: float64

In [5]:

#### 全量信息，输入为日期，输出为全量信息
def findOptionPrice(date):
    SQL_cmd = f"""SELECT instrument_id, trading_date, close, volume, open_interest,underlying_instr_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    """
    df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    df["underlying_instr_id"] = df["underlying_instr_id"].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )

    datebar = date[:4] + "-" + date[4:6] + "-" + date[6:]
    SQL_cmd = f"""SELECT instrument_id, trading_date, close
    FROM daily_data
    WHERE trading_date = '{datebar}'
    """
    unddf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    unddf["trading_date"] = unddf["trading_date"].apply(
        lambda x: x[:4] + x[5:7] + x[8:10]
    )
    
    df = pd.merge(
        df,
        unddf,
        left_on=["underlying_instr_id", "trading_date"],
        right_on=["instrument_id", "trading_date"],
        how="left",
    )
    df = df.rename(columns={"close_y": "underlying_close", "close_x": "option_close"})
    df = df.drop(columns=["instrument_id_y"])
    df = df.rename(columns={"instrument_id_x": "instrument_id"})

    info_dict = findUnderlyingOptionInfo(date)

    invalid_rows = []
    error_set = []
    # 遍历每一行
    for index, row in df.iterrows():
        try:
            prod, underlying_id, type_, k = parse_option_contract(row["instrument_id"])
            if (type_ == "c" and row["underlying_close"] > k) or (
                type_ == "p" and row["underlying_close"] < k
            ):
                invalid_rows.append(index)
                continue
            expiredate = info_dict[underlying_id]["expiredate"]
            exchange = info_dict[underlying_id]["exchange_id"]
            commission = info_dict[underlying_id]["open_money_by_vol"]
            margin_ratio = info_dict[underlying_id]["margin_ratio"]
            multi = info_dict[underlying_id]["volume_multiple"]
            tick = info_dict[underlying_id]["price_tick"]
            sector = info_dict[underlying_id]["sector"]
            updownlimit = info_dict[underlying_id]["updownlimit"]

            ttm = len(
                CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
            )
            df.at[index, "product"] = prod
            df.at[index, "sector"] = sector
            df.at[index, "opt_typ"] = type_
            df.at[index, "strike"] = k
            df.at[index, "tdtm"] = ttm
            df.at[index, "cdtm"] = (
                datetime.strptime(expiredate, "%Y%m%d")
                - datetime.strptime(date, "%Y%m%d")
            ).days
            df.at[index, "commission"] = commission
            df.at[index, "multiplier"] = multi
            df.at[index, "tick"] = tick
            df.at[index, "expiredate"] = expiredate
            df.at[index, "updownlimit"] = updownlimit
            df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
            iv = CCF.IV(
                row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
            )
            df.at[index, "iv"] = iv
            df.at[index, "delta"] = CCF.BSMgreeks.delta(
                type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
            )
            otmvalue = (
                row["underlying_close"] * multi - k * multi
                if type_ == "p"
                else k * multi - row["underlying_close"] * multi
            )
            otmvalue = max(otmvalue, 0)
            orimargin = row["underlying_close"] * multi * margin_ratio
            df.at[index, "margin"] = row["option_close"] * multi + max(
                orimargin - 0.5 * otmvalue, 0.5 * orimargin
            )
        except:
            # 若解析失败，记录行索引
            #             print(row['instrument_id'], ValueError)
            invalid_rows.append(index)
            error_set.append((index, row["instrument_id"], ValueError))
    df = df.drop(invalid_rows).dropna().reset_index(drop=True)
    df[["volume", "open_interest", "tdtm", "cdtm", "multiplier"]] = df[
        ["volume", "open_interest", "tdtm", "cdtm", "multiplier"]
    ].astype(int)

    return df


def parse_option_contract(contract):
    # 查找第一个数字的位置
    first_digit_index = next(
        (i for i, char in enumerate(contract) if char.isdigit()), None
    )
    if first_digit_index is None:
        raise ValueError(f"无法解析合约: {contract}，未找到数字。")
    product = contract[:first_digit_index]

    if "-" in contract:
        # 处理形如 a2505-C-3850 的合约
        parts = contract.split("-")
        underlying_id = parts[0]
        type_ = parts[1]
        k = float(parts[2])
    else:
        # 处理形如 zn2505P26000 的合约
        type_index = None
        for i in range(1, len(contract) - 1):
            if (
                contract[i].isalpha()
                and contract[i - 1].isdigit()
                and contract[i + 1].isdigit()
            ):
                type_index = i
                break

        if type_index is None:
            raise ValueError(f"无法解析合约: {contract}，请检查合约格式。")

        underlying_id = contract[:type_index]
        type_ = contract[type_index]
        k = float(contract[type_index + 1 :])
    
    # 修改 underlying_id
    if underlying_id.startswith('HO'):
        underlying_id = 'IH' + underlying_id[2:]
        product = 'IH'
    elif underlying_id.startswith('IO'):
        underlying_id = 'IF' + underlying_id[2:]
        product = 'IF'
    elif underlying_id.startswith('MO'):
        underlying_id = 'IM' + underlying_id[2:]
        product = 'IM'

    return product, underlying_id, type_.lower(), k


def findUnderlyingOptionInfo(date):
    SQL_cmd = f"""
    SELECT underlying_instr_id,instrument_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    ORDER BY open_interest DESC
    """
    max_open_interest = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    max_open_interest = (
        max_open_interest.groupby("underlying_instr_id").first().reset_index()
    )
    max_open_interest["underlying_instr_id"] = max_open_interest["underlying_instr_id"].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    SQL_cmd = f"""
    SELECT         
        i.underlying_instr_id,
        i.exchange_id,
        i.expiredate,
        i.volume_multiple,
        i.price_tick,
        icr.open_money_by_vol
    FROM instrument i
    LEFT JOIN commission_info icr ON i.instrument_id = icr.instrument_id
    WHERE i.instrument_id IN {tuple(max_open_interest['instrument_id'])}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info["underlying_instr_id"] = instrument_info["underlying_instr_id"].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    instrument_info['open_money_by_vol'] = np.where(instrument_info['underlying_instr_id'].str.startswith(('IH', 'IF', 'IM')), 15, instrument_info['open_money_by_vol'])
    instrument_info.to_excel('./ins.xlsx')
    SQL_cmd = f"""
    SELECT instrument_id as underlying_instr_id, short_margin_ratio as margin_ratio, product_id as product
    FROM instrument
    WHERE instrument_id IN {tuple(max_open_interest['underlying_instr_id'])}
    """
    margin_ratio = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    SQL_cmd = f"""
    SELECT PRODUCT_ID as product, WIND_INDUSTRYNAME1 as sector
    FROM WindIndustry
    WHERE PRODUCT_ID IN {tuple(margin_ratio['product'])}
    """
    secinfo = pd.read_sql(sql=SQL_cmd, con=CCF.research)
    secinfo.loc[len(secinfo)] = ['IH','金融']
    secinfo.loc[len(secinfo)] = ['IF','金融']
    secinfo.loc[len(secinfo)] = ['IC','金融']
    secinfo.loc[len(secinfo)] = ['IM','金融']
    SQL_cmd = f"""
    SELECT         
    instrument_id,
    prev_settlement,
    limit_up
    FROM daily_data
    WHERE trading_date = '{date[:4]}-{date[4:6]}-{date[6:]}'
    """
    limitupdf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    limitupdf["updownlimit"] = (
        limitupdf["limit_up"] / limitupdf["prev_settlement"] - 1
    ).round(2)
    limitupdf = limitupdf[["instrument_id", "updownlimit"]]
    limitupdf = limitupdf.rename(columns={"instrument_id": "underlying_instr_id"})

    merged_df = pd.merge(
        max_open_interest, instrument_info, on="underlying_instr_id", how="left"
    )
    merged_df = pd.merge(merged_df, margin_ratio, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, limitupdf, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, secinfo, on="product", how="left")
    merged_df = merged_df.drop(columns=["instrument_id"]).dropna()
    merged_dict = merged_df.set_index("underlying_instr_id").T.to_dict("dict")
    for key in merged_dict:
        if merged_dict[key]["exchange_id"] == "CFFEX":
            merged_dict[key]["exchange_id"] = "CFE"
        elif merged_dict[key]["exchange_id"] == "SHFE":
            merged_dict[key]["exchange_id"] = "SHF"
        elif merged_dict[key]["exchange_id"] == "CZCE":
            merged_dict[key]["exchange_id"] = "CZC"
        elif merged_dict[key]["exchange_id"] == "GFEX":
            merged_dict[key]["exchange_id"] = "GFE"
    return merged_dict


def getPayoffSeries(product, dtm, underlying_close, strike, opt_typ):
    pricedis = findRetDistribution(product, dtm) * underlying_close
    if opt_typ == "c":
        payoffser = pricedis - strike
    else:
        payoffser = strike - pricedis
    payoffser[payoffser < 0] = 0
    return payoffser

  0%|          | 0/10 [00:00<?, ?it/s]

NameError: name 'fund_etf_hist_em_df' is not defined

In [2]:
date = (dt.date.today()).strftime(format="%Y%m%d")
# date = "20250414"
# 读取期权价格数据
optdf = findOptionPrice(date)
# 设置期权价格数据的索引
optdf.index = optdf["instrument_id"]
# 调整保证金
optdf["margin"] = optdf["margin"] * 1.1
optdf["margin"] = optdf["margin"].round(0)
# 计算期权的收益
payoffmap = {}

def calculate_payoff(row):
    ins_id = row["instrument_id"]
    product = row["product"]
    dtm = row["tdtm"]
    underlying_close = row["underlying_close"]
    strike = row["strike"]
    opt_typ = row["opt_typ"]
    payoff = (
        getPayoffSeries(product, dtm, underlying_close, strike, opt_typ)
        * row["multiplier"]
    )
    payoffmap[ins_id] = payoff
    return pd.Series(
        [payoff.mean(), payoff.quantile(0.95), payoff.max(), len(payoff)]
    )
optdf['NumLimit'] = (np.abs(np.log(optdf['strike']/optdf['underlying_close']))/optdf['updownlimit']).round(2)
optdf['TradingValue'] = optdf['option_close'] * optdf['multiplier'] * optdf['volume']
optdf[["ExpectedPayoff", "Q95Payoff", "MaxPayoff", "DaysInSample"]] = optdf.apply(
    calculate_payoff, axis=1
)
# 计算期权的信用收取
optdf["CreditCollected"] = optdf["option_close"] * optdf["multiplier"]
# 计算期权的预期承保费
optdf["ExpectedLotPremium"] = optdf["CreditCollected"] - optdf["ExpectedPayoff"]
# 计算期权的理想收益
optdf["IdealRet(C)"] = (
    (optdf["CreditCollected"] - optdf["commission"])
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的预期边际收益
optdf["EMarginRet(C)"] = (
    (optdf["ExpectedLotPremium"] - optdf["commission"] * 2)
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的95%边际收益
optdf["95MarginRet(C)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["Q95Payoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)
# 计算期权的最差边际收益
optdf["WorstMarginRet(C)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["MaxPayoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 365
    / optdf["cdtm"]
)

optdf["IdealRet(T)"] = (
    (optdf["CreditCollected"] - optdf["commission"])
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的预期边际收益
optdf["EMarginRet(T)"] = (
    (optdf["ExpectedLotPremium"] - optdf["commission"] * 2)
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的95%边际收益
optdf["95MarginRet(T)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["Q95Payoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)
# 计算期权的最差边际收益
optdf["WorstMarginRet(T)"] = (
    (
        optdf["option_close"] * optdf["multiplier"]
        - optdf["MaxPayoff"]
        - optdf["commission"] * 2
    )
    / optdf["margin"]
    * 243
    / optdf["tdtm"]
)

# 删除空值
optdf = optdf.dropna()

# 复制一份期权数据，为筛选后结果
std_df = optdf.copy()
# 筛选出预期边际收益大于等于0.05的数据
std_df = std_df[std_df["EMarginRet(C)"] >= 0.05]
# 筛选出delta绝对值小于0.1的数据
std_df = std_df[np.abs(std_df["delta"]) < 0.1]
# 筛选出dtm小于等于20的数据
std_df = std_df[std_df["tdtm"] <= 20]
# 筛选出持仓量大于等于200的数据
std_df = std_df[std_df["open_interest"] >= 200]
# 筛选出最大收益为0且样本天数大于等于500的数据
std_df = std_df[(std_df["MaxPayoff"] == 0) & (std_df["DaysInSample"] >= 500)]



In [5]:
#  输入为一行optdf，输出该行在该交易日14:55时间点的ask和bid
def findOptionQuote(row):
    date = row['trading_date']
    ins = row['instrument_id']
    try: 
        row = CCF.Option_Raw_Data(ins, date).iloc[-601]
        return float(row['Ask1']), float(row['Bid1'])
    except:
        return np.nan, np.nan
std_df['bid'] = std_df.apply(lambda x: findOptionQuote(x)[1], axis = 1)
# 生成excel数据
exceldf = std_df[
    [
        "underlying_instr_id",
        "product",
        "option_close",
        "sector",
        "NumLimit",
        "EMarginRet(C)",
        "EMarginRet(T)",
        "margin",
        "tdtm",
        'bid'
    ]
].dropna().reset_index()
exceldf["EMarginRet(C)"] = (exceldf["EMarginRet(C)"] * 100).round(2)
exceldf["EMarginRet(T)"] = (exceldf["EMarginRet(T)"] * 100).round(2)
undrownum = exceldf.groupby("underlying_instr_id").size()
undmeanret = (
    exceldf.groupby("underlying_instr_id")["EMarginRet(T)"]
    .mean()
    .sort_values(ascending=False)
)
exceldf = exceldf.groupby(["underlying_instr_id", "instrument_id"]).first()
result = []
cnt = 0
sett = []
for und in undmeanret.index:
    thisnum = undrownum[und]
    if thisnum + cnt > 120:
        result.append(exceldf.loc[(sett, slice(None)), :])
        sett = [und]
        cnt = thisnum
    else:
        sett.append(und)
        cnt += thisnum
result.append(exceldf.loc[(sett, slice(None)), :])

# 生成json数据
jsondf = std_df.reset_index(drop=True)
jsonoutput = []
for ind, row in jsondf.iterrows():
    rowjs = {}
    idd = f"{date[2:]}#{ind + 1}"
    rowjs["_id"] = idd
    rowjs["idnum"] = ind + 1
    rowjs["create_date"] = date
    rowjs["underlying_id"] = row["underlying_instr_id"]
    rowjs["product_id"] = row["product"]
    rowjs["sector"] = row["sector"]
    rowjs["margin_req"] = row["margin"]
    rowjs["expire_date"] = row["expiredate"]
    rowjs["STG_TYP"] = ["NK_SELL"]
    rowjs["HEDG_MTD"] = ["StopLoss"]
    rowjs["option_structure"] = {row["instrument_id"]: -1}
    rowjs["risk_params"] = {
        "limit_price": row["option_close"] * -1,
        "ExpectedPayoff": row["ExpectedPayoff"],
        "EMarginRet": row["EMarginRet(T)"],
    }
    jsonoutput.append(rowjs)

# 保存json数据
direc = f"E:\\KurStrategy\\dailySummary\\{date}\\"
Path(direc).mkdir(parents=True, exist_ok=True)
with open(direc + f"{date}_jssave.json", "w") as f:
    json.dump(jsonoutput, f)
# 保存excel数据
optdf.to_excel(direc + f"{date}_totalSet.xlsx")
std_df.to_excel(direc + f"{date}_selectSet.xlsx")
# 生成excel图片
direc = f"E:\\KurStrategy\\dailySummary\\{date}\\picfile\\"
Path(direc).mkdir(parents=True, exist_ok=True)
for i in range(len(result)):
    dfi.export(
        result[i],
        f"{direc}{date}_selectSetSimp_{i+1}.png",
        fontsize=2,
        dpi=900,
        table_conversion="chrome",
        chrome_path="C:\Program Files\Google\Chrome Dev\Application\chrome.exe",
        max_rows=-1,
    )

In [11]:
std_df['underlying_instr_id'].unique()

array(['a2505', 'al2505', 'au2505', 'b2505', 'c2505', 'cs2505', 'cu2505',
       'eb2505', 'eg2505', 'FG506', 'i2505', 'jd2505', 'lh2505', 'm2505',
       'p2505', 'pp2505', 'ru2505', 'si2506', 'sn2505', 'v2505', 'y2505'],
      dtype=object)

In [4]:
exceldf

product  option_close sector  option_close  \
underlying_instr_id instrument_id                                              
FG506               FG506C1600         FG           1.0   能化建材           1.0   
                    FG506C1620         FG           1.0   能化建材           1.0   
                    FG506C1660         FG           1.0   能化建材           1.0   
                    FG506C1680         FG           0.5   能化建材           0.5   
                    FG506C1700         FG           0.5   能化建材           0.5   
...                                   ...           ...    ...           ...   
y2505               y2505-C-9000        y           0.5    农产品           0.5   
                    y2505-C-9200        y           0.5    农产品           0.5   
                    y2505-C-9400        y           0.5    农产品           0.5   
                    y2505-P-6400        y           0.5    农产品           0.5   
                    y2505-P-6900        y           0.5    农产品           0.5   

                                   EMarginRet(C)  EMarginRet(T)  margin  tdtm  
underlying_instr_id instrument_id                                              
FG506               FG506C1600             18.28          19.61  1308.0    18  
                    FG506C1620             18.28          19.61  1308.0    18  
                    FG506C1660             18.28          19.61  1308.0    18  
                    FG506C1680              8.73           9.37  1297.0    18  
                    FG506C1700              8.73           9.37  1297.0    18  
...                                          ...            ...     ...   ...  
y2505               y2505-C-9000           16.31          10.86  2983.0     3  
                    y2505-C-9200           16.31          10.86  2983.0     3  
                    y2505-C-9400           16.31          10.86  2983.0     3  
                    y2505-P-6400           16.31          10.86  2983.0     3  
                    y2505-P-6900           16.31          10.86  2983.0     3  

[141 rows x 8 columns]

In [8]:
dfi.export(
    exceldf,
    f"{direc}{date}_1234.png",
    fontsize=2,
    dpi=900,
    table_conversion='chrome',
    chrome_path='C:\Program Files\Google\Chrome Dev\Application\chrome.exe',
    max_rows=-1)

In [58]:
exceldf = std_df[['underlying_instr_id', 'product', 'option_close', 'sector', 'option_close', 'EMarginRet', 'margin', 'dtm']].reset_index()
exceldf['EMarginRet'] = (exceldf['EMarginRet'] * 100).round(2)
undrownum = exceldf.groupby('underlying_instr_id').size()
undmeanret = exceldf.groupby('underlying_instr_id')['EMarginRet'].mean().sort_values(ascending=False)
exceldf = exceldf.groupby(['underlying_instr_id', 'instrument_id']).first()
result = []
cnt = 0
sett = []
for und in undmeanret.index:
    thisnum = undrownum[und]
    if(thisnum + cnt > 120):
        result.append(exceldf.loc[(sett, slice(None)), :])
        sett = [und]
        cnt = thisnum
    else:
        sett.append(und)
        cnt += thisnum
result.append(exceldf.loc[(sett, slice(None)), :])

In [48]:
exceldf.loc[(['FG506', 'MA505'], slice(None)), :]

product  option_close sector  option_close  \
underlying_instr_id instrument_id                                              
FG506               FG506C1680         FG           1.5   能化建材           1.5   
                    FG506C1700         FG           0.5   能化建材           0.5   
MA505               MA505C2650         MA           0.5   能化建材           0.5   
                    MA505C2700         MA           0.5   能化建材           0.5   
                    MA505C2850         MA           0.5   能化建材           0.5   
                    MA505C2900         MA           0.5   能化建材           0.5   
                    MA505C3000         MA           0.5   能化建材           0.5   
                    MA505P2025         MA           0.5   能化建材           0.5   
                    MA505P2050         MA           0.5   能化建材           0.5   
                    MA505P2075         MA           0.5   能化建材           0.5   
                    MA505P2100         MA           0.5   能化建材           0.5   
                    MA505P2125         MA           0.5   能化建材           0.5   
                    MA505P2150         MA           0.5   能化建材           0.5   
                    MA505P2175         MA           0.5   能化建材           0.5   

                                   EMarginRet  margin  dtm  
underlying_instr_id instrument_id                           
FG506               FG506C1680          26.57  1326.0   20  
                    FG506C1700           8.39  1304.0   20  
MA505               MA505C2650          90.59  1073.0    1  
                    MA505C2700          90.59  1073.0    1  
                    MA505C2850          90.59  1073.0    1  
                    MA505C2900          90.59  1073.0    1  
                    MA505C3000          90.59  1073.0    1  
                    MA505P2025          90.59  1073.0    1  
                    MA505P2050          90.59  1073.0    1  
                    MA505P2075          90.59  1073.0    1  
                    MA505P2100          90.59  1073.0    1  
                    MA505P2125          90.59  1073.0    1  
                    MA505P2150          90.59  1073.0    1  
                    MA505P2175          90.59  1073.0    1

In [56]:
result

In [59]:
result[2]

product  option_close    sector  \
underlying_instr_id instrument_id                                   
PK505               PK505C8600         PK           0.5       农产品   
                    PK505C8700         PK           0.5       农产品   
                    PK505C8800         PK           0.5       农产品   
                    PK505C8900         PK           0.5       农产品   
                    PK505C9300         PK           0.5       农产品   
                    PK505P7400         PK           0.5       农产品   
                    PK505P7500         PK           0.5       农产品   
                    PK505P7600         PK           0.5       农产品   
ru2505              ru2505C18500       ru           5.0      能化建材   
                    ru2505C18750       ru           5.0      能化建材   
                    ru2505C19000       ru           5.0      能化建材   
                    ru2505C19250       ru           4.0      能化建材   
                    ru2505C19500       ru           4.0      能化建材   
                    ru2505C20000       ru           3.0      能化建材   
                    ru2505C20250       ru           4.0      能化建材   
                    ru2505C20750       ru           4.0      能化建材   
                    ru2505C21500       ru           4.0      能化建材   
                    ru2505C21750       ru           4.0      能化建材   
i2505               i2505-C-1000        i           0.1   煤焦钢矿黑色系   
                    i2505-C-1040        i           0.1   煤焦钢矿黑色系   
                    i2505-C-950         i           0.1   煤焦钢矿黑色系   
                    i2505-C-960         i           0.1   煤焦钢矿黑色系   
                    i2505-C-970         i           0.1   煤焦钢矿黑色系   
                    i2505-P-540         i           0.2   煤焦钢矿黑色系   
                    i2505-P-560         i           0.1   煤焦钢矿黑色系   
                    i2505-P-580         i           0.2   煤焦钢矿黑色系   
rb2505              rb2505C3850        rb           1.0   煤焦钢矿黑色系   
                    rb2505C3950        rb           1.0   煤焦钢矿黑色系   
                    rb2505C4050        rb           1.0   煤焦钢矿黑色系   
                    rb2505C4100        rb           1.0   煤焦钢矿黑色系   
v2505               v2505-C-5900        v           0.5      能化建材   
                    v2505-C-6000        v           0.5      能化建材   
                    v2505-C-6100        v           0.5      能化建材   
                    v2505-C-6200        v           0.5      能化建材   
                    v2505-C-7600        v           0.5      能化建材   
eg2505              eg2505-C-5200      eg           0.5      能化建材   
                    eg2505-C-5300      eg           0.5      能化建材   
                    eg2505-C-5400      eg           0.5      能化建材   
al2505              al2505C23400       al           3.0  贵金属与有色金属   
l2505               l2505-C-8300        l           0.5      能化建材   

                                   option_close  EMarginRet  margin  dtm  
underlying_instr_id instrument_id                                         
PK505               PK505C8600              0.5        9.72  2249.0    1  
                    PK505C8700              0.5       11.08  1974.0    1  
                    PK505C8800              0.5       12.22  1789.0    1  
                    PK505C8900              0.5       12.22  1789.0    1  
                    PK505C9300              0.5       12.22  1789.0    1  
                    PK505P7400              0.5       12.22  1789.0    1  
                    PK505P7500              0.5       11.66  1875.0    1  
                    PK505P7600              0.5       10.17  2150.0    1  
ru2505              ru2505C18500            5.0       13.05  8190.0   10  
                    ru2505C18750            5.0       13.05  8190.0   10  
                    ru2505C19000            5.0       13.05  8190.0   10  
                    ru2505C19250            4.0       10.10  8179.0   10  
                    ru2505C19500            4.0       10.10  8179.0   10  
          

In [3]:
zxtz = pd.read_excel('./zxtz.xlsx')
zzsp = pd.read_excel('./zzsp.xlsx')
zzsp = zzsp[['future_id','weight']]
SQL_cmd = f"""
SELECT         
PRODUCT_ID, simpleRY
FROM RY_FULL
WHERE TRADE_DATE = '20250414'
"""
rydf = pd.read_sql(sql=SQL_cmd, con=CCF.research)
def findproduct(strr):
    for i in range(len(strr)):
        if(strr[i].isdigit()):
            return strr[:i]
zxtz['PRODUCT_ID'] = zxtz['future_id'].apply(lambda x: findproduct(x))
zzsp['PRODUCT_ID'] = zzsp['future_id'].apply(lambda x: findproduct(x))
info = findUnderlyingOptionInfo('20250411')
secinfo = {}
for i in info:
    secinfo[info[i]['product']] = info[i]['sector']
mergedf = pd.merge(zxtz, zzsp, on='PRODUCT_ID', how='outer', suffixes=('_招享通胀', '_中证商品'))[['PRODUCT_ID','sector','weight_招享通胀','weight_中证商品']].fillna(0)
mergedf['sector'] = mergedf['PRODUCT_ID'].apply(lambda x: secinfo[x] if x in secinfo else '煤焦钢矿黑色系')
mergedf = pd.merge(mergedf, rydf, on='PRODUCT_ID', how='left')

In [25]:
mergedf.to_excel(f'./weightinfo.xlsx')